# Train YOLOv11s on Kaggle P100 (with auto-save callback)

**Khác với Colab notebook:**
- Auto-save best.pt mỗi epoch về Kaggle dataset output (không bao giờ mất)
- Reduce epochs 100 → 70 (đã thấy plateau từ epoch 75 trên Colab)
- Better resume nếu disconnect

## Setup Kaggle

1. New Notebook → Settings → Accelerator: **GPU P100** (bắt buộc)
2. Add Data: upload `yolo_dataset_merged.zip` (~547MB) — sẽ ở `/kaggle/input/`
3. Save Version → Save & Run All (khi muốn run từ đầu)

## 1. Verify GPU

In [1]:
!nvidia-smi

Thu Apr 30 04:50:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install ultralytics

In [2]:
!pip install -q ultralytics 'numpy<2.1'
import ultralytics
ultralytics.checks()

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6841.9/8062.4 GB disk)


## 3. Extract dataset từ Kaggle Input

In [4]:
import os, glob

# Kaggle đã auto-extract zip rồi → tìm data.yaml trực tiếp
data_yaml_paths = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
print('Found data.yaml:', data_yaml_paths)

if not data_yaml_paths:
    # Fallback: nếu chưa extract, extract từ zip
    import zipfile
    zip_files = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    print('Zip files:', zip_files)
    EXTRACT_TO = '/kaggle/working/dataset'    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(zip_files[0], 'r') as zf:
        zf.extractall(EXTRACT_TO)
    data_yaml_paths = glob.glob(f'{EXTRACT_TO}/**/data.yaml', recursive=True)

DATA_YAML = data_yaml_paths[0]
print(f'\ndata.yaml: {DATA_YAML}')
!cat {DATA_YAML}


Found data.yaml: ['/kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/data.yaml']

data.yaml: /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/data.yaml
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 12
names:
- physical_damage
- gen_11
- gen_12_13
- gen_14
- gen_15
- gen_16
- gen_17
- gen_6
- gen_7_8
- gen_x_xs
- scratch
- screen_defect
roboflow:
  workspace: thngs-workspace-akqw8
  project: iphone-pricing-detection
  version: 2
  license: Private
  url: https://app.roboflow.com/thngs-workspace-akqw8/iphone-pricing-detection/2


## 4. Fix paths

In [13]:
import yaml
import os
import glob

# Auto-find data.yaml (Kaggle có thể đặt tên folder khác)
yaml_paths = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
if not yaml_paths:
    raise FileNotFoundError("Không tìm thấy data.yaml trong /kaggle/input/")
ORIG_YAML = yaml_paths[0]
DATASET_ROOT = os.path.dirname(ORIG_YAML)
print(f'ORIG_YAML: {ORIG_YAML}')
print(f'DATASET_ROOT: {DATASET_ROOT}')

with open(ORIG_YAML) as f:
    cfg = yaml.safe_load(f)

# Format chuẩn ultralytics
cfg['path']  = DATASET_ROOT
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'

# Save sang /kaggle/working (writable)
DATA_YAML = '/kaggle/working/data.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nClasses ({cfg['nc']}): {cfg['names']}")
print(f"\nNew yaml: {DATA_YAML}")

print('\n=== Verify image folders ===')
for split in ['train', 'valid', 'test']:
    p = os.path.join(cfg['path'], split, 'images')
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    status = '✓' if n > 0 else '✗ MISSING'
    print(f'{status} {split}: {p} → {n} files')


ORIG_YAML: /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/data.yaml
DATASET_ROOT: /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged

Classes (12): ['physical_damage', 'gen_11', 'gen_12_13', 'gen_14', 'gen_15', 'gen_16', 'gen_17', 'gen_6', 'gen_7_8', 'gen_x_xs', 'scratch', 'screen_defect']

New yaml: /kaggle/working/data.yaml

=== Verify image folders ===
✓ train: /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/train/images → 10359 files
✓ valid: /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/valid/images → 987 files
✓ test: /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/test/images → 492 files


## 5. Auto-save callback — bảo hiểm chống mất train

Sau MỖI epoch, copy best.pt sang `/kaggle/working/` (sẽ tự động commit khi save notebook version).

In [14]:
import shutil
from ultralytics import YOLO
from ultralytics.utils import callbacks

BACKUP_DIR = '/kaggle/working/model_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

def on_fit_epoch_end(trainer):
    """Sau mỗi epoch, copy best.pt + last.pt sang BACKUP_DIR."""
    weights_dir = trainer.save_dir / 'weights'
    for fname in ['best.pt', 'last.pt']:
        src = weights_dir / fname
        if src.exists():
            shutil.copy(src, f'{BACKUP_DIR}/{fname}')
    epoch = trainer.epoch + 1
    metrics = trainer.metrics if trainer.metrics else {}
    print(f'[BACKUP] Epoch {epoch} → {BACKUP_DIR}/best.pt | mAP50={metrics.get("metrics/mAP50(B)", 0):.4f}')

print('Callback registered. Ready to train.')

Callback registered. Ready to train.


## 6. Train YOLOv11s — 70 epochs (đã learn từ lần trước)

Dựa trên Colab run trước:
- mAP@50 epoch 80 = 0.582 (plateau từ epoch 75)
- Patience=15 đủ early stop nếu plateau
- 70 epochs trên P100 ≈ **2-2.5 giờ** (P100 nhanh hơn T4)

In [15]:
model = YOLO('yolo11s.pt')

# Register backup callback
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

results = model.train(
    data=DATA_YAML,
    epochs=70,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    cos_lr=True,
    patience=15,
    cache=True,
    amp=True,
    device=0,
    project='/kaggle/working/runs',
    name='yolov11s_iphone',
    exist_ok=True,
)

print('\nTraining done!')
print(f'Best weights: {results.save_dir}/weights/best.pt')
print(f'Backup: {BACKUP_DIR}/best.pt')

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11s_iphone, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=15,

## 7. Evaluate test set

In [16]:
best_pt = f'{BACKUP_DIR}/best.pt'
model = YOLO(best_pt)

metrics = model.val(data=DATA_YAML, split='test', imgsz=640)
print(f'\n=== TEST SET METRICS ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'\nPer-class mAP@50:')
for i, name in enumerate(cfg['names']):
    print(f'  {name:<20} {metrics.box.maps[i]:.4f}')

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,417,444 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 3.8±2.2 ms, read: 5.3±1.0 MB/s, size: 43.8 KB)
val: Scanning /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/test/labels... 492 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 492/492 138.5it/s 3.6s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/lvntiu/yolo-iphone-dataset/yolo_dataset_merged/test is not writable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 135, len(boxes) = 923. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 3.9it/s 8.0s0.2s
                   all      

## 8. Final save — copy về /kaggle/working/ (auto-persist when commit)

Khi click **Save Version → Save & Run All** ở Kaggle, mọi file trong `/kaggle/working/` được commit vào output dataset của notebook → có thể download bất cứ lúc nào.

In [17]:
FINAL_OUT = '/kaggle/working/final'
os.makedirs(FINAL_OUT, exist_ok=True)

# Copy best/last
shutil.copy(f'{BACKUP_DIR}/best.pt', f'{FINAL_OUT}/best.pt')
shutil.copy(f'{BACKUP_DIR}/last.pt', f'{FINAL_OUT}/last.pt')

# Copy graphs
src_dir = '/kaggle/working/runs/yolov11s_iphone'
for fname in ['results.png', 'results.csv', 'confusion_matrix.png',
              'confusion_matrix_normalized.png', 'F1_curve.png',
              'PR_curve.png', 'P_curve.png', 'R_curve.png']:
    src = f'{src_dir}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{FINAL_OUT}/{fname}')

print(f'Saved to: {FINAL_OUT}')
!ls -la {FINAL_OUT}
print('\n👉 Click "Save Version" ở Kaggle để commit output')

Saved to: /kaggle/working/final
total 38328
drwxr-xr-x 2 root root     4096 Apr 30 09:06 .
drwxr-xr-x 6 root root     4096 Apr 30 09:06 ..
-rw-r--r-- 1 root root 19188186 Apr 30 09:06 best.pt
-rw-r--r-- 1 root root   320106 Apr 30 09:06 confusion_matrix_normalized.png
-rw-r--r-- 1 root root   265086 Apr 30 09:06 confusion_matrix.png
-rw-r--r-- 1 root root 19188186 Apr 30 09:06 last.pt
-rw-r--r-- 1 root root     8772 Apr 30 09:06 results.csv
-rw-r--r-- 1 root root   254876 Apr 30 09:06 results.png

👉 Click "Save Version" ở Kaggle để commit output
